# EV-Szenario: Ladeinfrastruktur Edeka-Parkplatz

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandapower as pp
import itertools
import json

## 1. Daten laden

In [ ]:
KABEL_PATH    = "data/netz_excels/kabel_excel.xlsx"
KNOTEN_PATH   = "data/netz_excels/knoten_excel.xlsx"
PARAMS_PATH   = "data/netz_excels/kabel_parameter.csv"
LASTEN_PATH   = "data/gebaudelasten/gebauedelasten.json"
FFE_PATH      = "data/e_ladelasten/EV_Lastprofile_gefiltert.csv"

kabel  = pd.read_excel(KABEL_PATH,  index_col=0)
knoten = pd.read_excel(KNOTEN_PATH, index_col=0)
params = pd.read_csv(PARAMS_PATH)

datetime_index = pd.date_range(start="2018-01-01", end="2018-12-31 23:00", freq="h")
with open(LASTEN_PATH, "r") as f:
    load_df = pd.DataFrame(json.load(f), index=datetime_index)

# Echte kW-Ladeprofile laden (191 gefilterte Ladevorgaenge <= 24 kW)
df_filtered = pd.read_csv(FFE_PATH, index_col=0)
ladevorgaenge_kw = []
for col in df_filtered.columns:
    profil = df_filtered[col].dropna().values
    if len(profil) > 0 and profil.max() > 0:
        ladevorgaenge_kw.append(profil)

print(f"Knoten: {len(knoten)}, Kabel: {len(kabel)}, Zeitschritte: {len(load_df)}")
print(f"Ladeprofile geladen: {len(ladevorgaenge_kw)}")

## 2. Netz aufbauen

In [ ]:
net = pp.create_empty_network(f_hz=50, sn_mva=10)

# Buses
for idx, row in knoten.iterrows():
    pp.create_bus(net, vn_kv=0.4, name=row["IDENTNUMME"], index=idx)

# MS-Sammelschiene und Trafo (630 kVA ONT)
bus_ms = pp.create_bus(net, vn_kv=20.0, name="MS_Sammelschiene")
pp.create_ext_grid(net, bus=bus_ms, vm_pu=1.0, name="Einspeisung")
pp.create_transformer_from_parameters(
    net,
    hv_bus=bus_ms, lv_bus=1,
    sn_mva=0.63,
    vn_hv_kv=20.0, vn_lv_kv=0.4,
    vkr_percent=1.1, vk_percent=4.0,
    pfe_kw=1.3, i0_percent=0.27,
    name="ONT_630kVA"
)

# Kabelparameter
params_dict = {
    (row["kabeltyp"], row["adern"], row["querschnitt_mm2"]): (
        row["r_ohm_per_km"], row["x_ohm_per_km"], row["c_nf_per_km"], row["max_i_ka"]
    )
    for _, row in params.iterrows()
}

# Leitungen
for idx, row in kabel.iterrows():
    key = (row["KABELTYP"], row["ADERN"], row["QUERSCHNIT"])
    if key not in params_dict:
        continue
    r, x, c, imax = params_dict[key]
    pp.create_line_from_parameters(
        net,
        from_bus=row["start_bus"], to_bus=row["end_bus"],
        length_km=row["length_m"] / 1000,
        r_ohm_per_km=r, x_ohm_per_km=x, c_nf_per_km=c, max_i_ka=imax,
        name=f"Kabel_{idx}"
    )

# Gebaeudelasten
for col in load_df.columns:
    bus_idx = int(col.split("_")[1])
    pp.create_load(net, bus=bus_idx, p_mw=load_df.iloc[0][col]/1000, q_mvar=0.0, name=f"Last_{bus_idx}")

# Parkplatz-Bus und Zuleitung
bus_parkplatz = pp.create_bus(net, vn_kv=0.4, name="Parkplatz_Edeka")
r, x, c, imax = params_dict[("NAYY", 4, 150)]
pp.create_line_from_parameters(
    net,
    from_bus=1, to_bus=bus_parkplatz,
    length_km=0.020,
    r_ohm_per_km=r, x_ohm_per_km=x, c_nf_per_km=c, max_i_ka=imax,
    name="Kabel_Parkplatz"
)
pp.create_load(net, bus=bus_parkplatz, p_mw=0.0, q_mvar=0.0, name="EV_Parkplatz")

leitung_parkplatz = net.line[net.line["name"] == "Kabel_Parkplatz"].index[0]

print(f"Buses: {len(net.bus)}, Leitungen: {len(net.line)}, Lasten: {len(net.load)}")
print(f"Parkplatz-Bus: {bus_parkplatz}, Parkplatz-Leitung: {leitung_parkplatz}")

# Testlauf
pp.runpp(net, algorithm='nr', numba=False)
print(f"Trafo-Auslastung (Testlauf): {net.res_trafo['loading_percent'].values[0]:.1f}%")

## 3. Stochastische EV-Lastprofil-Funktion

In [ ]:
def erstelle_ev_lastprofil(n_saeulen, g, ladevorgaenge_kw, datetime_index, seed=42):
    """
    Stochastisches EV-Lastprofil fuer ein Jahr (8760 h).
    - Anzahl Ankuenfte pro Tag: Poisson(lambda = n_saeulen * g), gedeckelt auf n_saeulen
    - Ankunftszeit: gleichverteilt 8-19 Uhr
    - Ladeprofil: echte kW-Messwerte (5-Min), auf Stunden gemittelt
    Returns: pd.Series mit stuendlicher Last in MW
    """
    rng = np.random.default_rng(seed)
    ev_last_mw = np.zeros(len(datetime_index))
    tage = pd.date_range("2018-01-01", "2018-12-31", freq="D")

    for tag in tage:
        n_ankuenfte = min(rng.poisson(lam=n_saeulen * g), n_saeulen)

        for _ in range(n_ankuenfte):
            ankunft_h = rng.integers(8, 20)
            profil    = ladevorgaenge_kw[rng.integers(0, len(ladevorgaenge_kw))]

            # 5-Minuten-Werte auf Stunden aggregieren (12 Werte pro Stunde)
            n_stunden = max(1, int(np.ceil(len(profil) / 12)))
            for h in range(n_stunden):
                start     = h * 12
                ende      = min(start + 12, len(profil))
                kw_stunde = np.mean(profil[start:ende])

                ts = tag + pd.Timedelta(hours=ankunft_h + h)
                if ts in datetime_index:
                    ev_last_mw[datetime_index.get_loc(ts)] += kw_stunde / 1000

    return pd.Series(ev_last_mw, index=datetime_index)

## 4. Haushaltsspitzenlastfall bestimmen

In [ ]:
haushaltslast_gesamt = load_df.sum(axis=1)  # kW
t_peak = haushaltslast_gesamt.idxmax()

print(f"Spitzenlast-Zeitpunkt: {t_peak}")
print(f"Gesamtlast Haushalte:  {haushaltslast_gesamt[t_peak]:.1f} kW")

# Haushaltslast einmalig setzen — bleibt fuer alle Szenarien gleich
for col in load_df.columns:
    bus_idx = int(col.split("_")[1])
    net.load.loc[net.load["bus"] == bus_idx, "p_mw"] = load_df.loc[t_peak, col] / 1000

## 5. Szenarienmatrix
- n_saeulen: 10, 20, 30, 40, 50
- g: 0.1, 0.25, 0.5, 0.8
- EV-Spitzenlast: stochastisch aus echten kW-Profilen (seed=42)

In [ ]:
n_saeulen_liste = [10, 20, 30, 40, 50]
g_liste         = [0.1, 0.25, 0.5, 0.8]

ergebnisse = []

for n_saeulen, g in itertools.product(n_saeulen_liste, g_liste):

    # Stochastisches EV-Profil — Jahresspitzenwert als worst case
    ev_last = erstelle_ev_lastprofil(
        n_saeulen=n_saeulen, g=g,
        ladevorgaenge_kw=ladevorgaenge_kw,
        datetime_index=datetime_index, seed=42
    )
    p_ev_peak_mw = ev_last.max()

    # EV-Spitzenlast ins Netz
    net.load.loc[net.load["name"] == "EV_Parkplatz", "p_mw"] = p_ev_peak_mw

    # Lastfluss
    pp.runpp(net, algorithm="nr", numba=False)

    ergebnisse.append({
        "n_saeulen"       : n_saeulen,
        "g"               : g,
        "p_ev_kw"         : p_ev_peak_mw * 1000,
        "vm_min_pu"       : net.res_bus["vm_pu"].min(),
        "loading_max_pct" : net.res_line["loading_percent"].max(),
        "loading_pk_pct"  : net.res_line.loc[leitung_parkplatz, "loading_percent"],
        "trafo_pct"       : net.res_trafo["loading_percent"].values[0],
        "leitung_ok"      : net.res_line["loading_percent"].max() <= 100.0,
        "trafo_ok"        : net.res_trafo["loading_percent"].values[0] <= 100.0,
        "spannung_ok"     : net.res_bus["vm_pu"].min() >= 0.95,
    })

    print(f"✅ {n_saeulen:2d} Saeulen, g={g} → "
          f"EV-Peak: {p_ev_peak_mw*1000:6.2f} kW, "
          f"Loading: {net.res_line['loading_percent'].max():5.1f}%, "
          f"Trafo: {net.res_trafo['loading_percent'].values[0]:5.1f}%, "
          f"Vm_min: {net.res_bus['vm_pu'].min():.4f} pu")

matrix_df = pd.DataFrame(ergebnisse)
print("\nFertig!")

## 6. Auswertung — Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, titel in zip(
    axes,
    ["loading_max_pct", "trafo_pct"],
    ["Max. Leitungsauslastung [%]", "Trafo-Auslastung [%]"]
):
    pivot = matrix_df.pivot(index="g", columns="n_saeulen", values=col)
    im = ax.imshow(pivot.values, cmap="RdYlGn_r", vmin=0, vmax=100, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel("Anzahl Ladesaeulen")
    ax.set_ylabel("Gleichzeitigkeitsfaktor g")
    ax.set_title(titel)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f"{pivot.values[i,j]:.0f}%",
                    ha="center", va="center", fontsize=9, fontweight="bold")
    plt.colorbar(im, ax=ax)

plt.suptitle(
    f"Szenarienmatrix — Spitzenlastfall\n"
    f"Haushalte: {t_peak.strftime('%d.%m.%Y %H:%M')} ({haushaltslast_gesamt[t_peak]:.1f} kW) + stochastischer EV-Peak (seed=42)",
    fontsize=11
)
plt.tight_layout()
plt.show()

## 7. Jahressimulation — Maximalszenario (50 Säulen, g=0.8)

In [ ]:
n_saeulen = 50
g         = 0.8

# Stochastisches EV-Profil fuer das gesamte Jahr
ev_last_jahr = erstelle_ev_lastprofil(
    n_saeulen=n_saeulen, g=g,
    ladevorgaenge_kw=ladevorgaenge_kw,
    datetime_index=datetime_index, seed=42
)

print(f"EV-Peak: {ev_last_jahr.max()*1000:.1f} kW")

# Jahressimulation
res_vm    = {}
res_load  = {}
res_trafo = {}

for i, (ts, row) in enumerate(load_df.iterrows()):
    for col in load_df.columns:
        bus_idx = int(col.split("_")[1])
        net.load.loc[net.load["bus"] == bus_idx, "p_mw"] = row[col] / 1000

    net.load.loc[net.load["name"] == "EV_Parkplatz", "p_mw"] = ev_last_jahr.iloc[i]

    pp.runpp(net, algorithm="nr", numba=False)

    res_vm[ts]    = net.res_bus["vm_pu"].copy()
    res_load[ts]  = net.res_line["loading_percent"].copy()
    res_trafo[ts] = net.res_trafo["loading_percent"].values[0]

vm_pu_df   = pd.DataFrame.from_dict(res_vm,   orient="index")
loading_df = pd.DataFrame.from_dict(res_load, orient="index")
trafo_df   = pd.Series(res_trafo)

print(f"Max. Leitungsauslastung: {loading_df.max().max():.1f}%")
print(f"Max. Trafo-Auslastung:   {trafo_df.max():.1f}%")
print(f"Min. Busspannung:        {vm_pu_df.min().min():.4f} pu")

## 8. Jahresdauerlinien

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Leitungsauslastung
for col in loading_df.columns:
    axes[0].plot(range(8760), loading_df[col].sort_values(ascending=False).values,
                 color="steelblue", linewidth=0.5, alpha=0.3)
axes[0].plot(range(8760), loading_df[leitung_parkplatz].sort_values(ascending=False).values,
             color="red", linewidth=1.5, label=f"Leitung {leitung_parkplatz} (Parkplatz-Zuleitung)")
axes[0].axhline(100, color="black", linestyle="--", linewidth=1.0, label="Grenzwert 100%")
axes[0].set_ylabel("Auslastung [%]")
axes[0].set_title(f"Jahresdauerlinie Leitungsauslastung — {n_saeulen} Saeulen, g={g}")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Trafo-Auslastung
axes[1].plot(range(8760), trafo_df.sort_values(ascending=False).values,
             color="darkorange", linewidth=1.5, label="Trafo 630 kVA")
axes[1].axhline(100, color="black", linestyle="--", linewidth=1.0, label="Grenzwert 100%")
axes[1].set_ylabel("Auslastung [%]")
axes[1].set_title("Jahresdauerlinie Trafo-Auslastung")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Busspannungen
for col in vm_pu_df.columns:
    axes[2].plot(range(8760), vm_pu_df[col].sort_values(ascending=True).values,
                 color="steelblue", linewidth=0.5, alpha=0.3)
axes[2].plot(range(8760), vm_pu_df[bus_parkplatz].sort_values(ascending=True).values,
             color="red", linewidth=1.5, label=f"Bus {bus_parkplatz} (Parkplatz)")
axes[2].axhline(0.95, color="black", linestyle="--", linewidth=1.0, label="Grenzwert 0.95 pu")
axes[2].set_xlabel("Stunden [h/a]")
axes[2].set_ylabel("Spannung [pu]")
axes[2].set_title("Jahresdauerlinie Busspannungen")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()